<a href="https://colab.research.google.com/github/Colanimmy/AAI2025-DEVIN-COPY/blob/2026fall/Coding_Exercise_ML_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# Load the real housing dataset
df = pd.read_csv("kc_house_data.csv")

# Keep the three fields required for Part 1:
# house price, square footage, and location (ZIP code)
df = df[["price", "sqft_living", "zipcode"]].dropna()

# Treat ZIP code as a categorical location rather than a numeric measurement
df["zipcode"] = df["zipcode"].astype(str)

# Features and target
X = df[["sqft_living", "zipcode"]]
y = df["price"]

# One-hot encode location and keep square footage as a numeric feature
preprocessor = ColumnTransformer(
    transformers=[
        ("location", OneHotEncoder(handle_unknown="ignore"), ["zipcode"])
    ],
    remainder="passthrough"
)

# Create the preprocessing + linear regression pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Predict the price of a 2,000 sq ft house in ZIP code 98103
new_house = pd.DataFrame({
    "sqft_living": [2000],
    "zipcode": ["98103"]
})
predicted_price = model.predict(new_house)[0]

print(f"Number of houses used: {len(df):,}")
print(f"Predicted price for a 2,000 sq ft house in ZIP code 98103: ${predicted_price:,.2f}")
print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"R-squared: {r2:.3f}")

# Display model coefficients
encoder = model.named_steps["preprocessor"].named_transformers_["location"]
feature_names = encoder.get_feature_names_out(["zipcode"]).tolist() + ["sqft_living"]
coefficients = model.named_steps["regressor"].coef_

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:,.2f}")

# The square-footage coefficient is the estimated price increase per extra sq ft,
# while holding ZIP code constant.
sqft_coefficient = coefficients[-1]
print(f"\nEstimated price increase per additional square foot: ${sqft_coefficient:,.2f}")

FileNotFoundError: [Errno 2] No such file or directory: 'kc_house_data.csv'